In [ ]:
!pip install -q transformers accelerate sentencepiece scipy pandas statsmodels

In [ ]:
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cosine

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [ ]:
tiny_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading TinyLlama tokenizer...")

tiny_tokenizer = AutoTokenizer.from_pretrained(
    tiny_id
)

print("Loading TinyLlama model...")

tiny_model = AutoModelForCausalLM.from_pretrained(
    tiny_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("TinyLlama loaded successfully.")

Loading TinyLlama tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading TinyLlama model...


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama loaded successfully.


In [ ]:
qwen_id = "Qwen/Qwen1.5-1.8B-Chat"

print("Loading Qwen tokenizer...")

qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_id
)

print("Loading Qwen model...")

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen loaded successfully.")

Loading Qwen tokenizer...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading Qwen model...


model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

Qwen loaded successfully.


In [ ]:
def extract_pooled_hidden_states(
    text,
    model,
    tokenizer,
    scramble=False
):

    if scramble:

        raw_ids = tokenizer.encode(
            text,
            add_special_tokens=False
        )

        random.shuffle(raw_ids)

        if tokenizer.bos_token_id is not None:

            input_ids = [
                [tokenizer.bos_token_id] + raw_ids
            ]

        else:

            input_ids = [raw_ids]

        inputs = {
            "input_ids": torch.tensor(
                input_ids
            ).to(model.device)
        }

    else:

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(model.device)

    with torch.no_grad():

        outputs = model(
            **inputs,
            output_hidden_states=True
        )

    pooled_layers = []

    for layer in outputs.hidden_states:

        pooled_vector = torch.mean(
            layer[0],
            dim=0
        )

        pooled_layers.append(
            pooled_vector
            .detach()
            .cpu()
            .to(torch.float32)
            .numpy()
        )

    return np.array(pooled_layers)

print("Extraction function ready.")

Extraction function ready.


In [ ]:
# Elite Rigor Stimuli Matrix: 20 Culturally Asymmetric Concepts
# Fully balanced across lengths and control paradigms

refined_comprehensive_dataset = [
    # --- DOMAIN 1: INTERPERSONAL OBLIGATIONS & RELATIONAL DYNAMICS ---
    {
        "id": 1, "domain": "Relational", "concept": "Giri (義理)",
        "E_base": "I am attending my co-worker's wedding because I want to support them as a close friend.",
        "J_asym": "同僚の結婚式に出席するのは、今後の良好な関係を保つための社会的な義理を果たすためだ。",
        "E_lit": "Attending a co-worker's wedding is to fulfill a social obligation to maintain good future relationships.",
        "S_sym": "Asisto a la boda de mi compañero de trabajo porque quiero apoyarlo como un amigo cercano."
    },
    {
        "id": 2, "domain": "Relational", "concept": "Ninjo (人情)",
        "E_base": "The manager chose to give the struggling employee another chance because of human empathy.",
        "J_asym": "マネージャーは、人間的な人情から、苦境にある従業員に、もう一度チャンスを与えることにした。",
        "E_lit": "The manager decided to give the struggling employee another chance out of humane empathy.",
        "S_sym": "El gerente decidió darle otra oportunidad al empleado en problemas debido a la empatía humana."
    },
    {
        "id": 3, "domain": "Relational", "concept": "En (縁)",
        "E_base": "We met unexpectedly in a distant city, and our successful partnership felt like a fortunate coincidence.",
        "J_asym": "私たちは遠く離れた街で偶然出会い、成功したパートナーシップは不思議な縁のように感じられた。",
        "E_lit": "We met by chance in a distant city, and our successful partnership felt like a mysterious karmic connection.",
        "S_sym": "Nos encontramos inesperadamente en una ciudad lejana, y nuestra exitosa asociación se sintió como una coincidencia afortunada."
    },
    {
        "id": 4, "domain": "Relational", "concept": "Amaeru (甘える)",
        "E_base": "The young child stayed close to their parents, depending completely on their deep affection and care.",
        "J_asym": "幼い子供は、親の深い愛情と保護に完全に依存し、遠慮なく甘えるような行動をとった。",
        "E_lit": "The young child depended completely on their parents' deep affection and protection, behaving responsively to presume upon their benevolence.",
        "S_sym": "El niño pequeño permaneció cerca de sus padres, dependiendo completamente de su profundo afecto y cuidado."
    },
    {
        "id": 5, "domain": "Relational", "concept": "Omoiyari (思いやり)",
        "E_base": "She quietly adjusted the room lighting before her tired roommate woke up, showing genuine kindness.",
        "J_asym": "彼女は疲れた同居人が起きる前に、相手を気遣う思いやりから、静かに部屋の照明を調整した。",
        "E_lit": "Out of altruistic consideration, she quietly adjusted the room lighting before her tired roommate woke up.",
        "S_sym": "Ella ajustó silenciosamente la iluminación de la habitación antes de que su compañero cansado se despertara, mostrando amabilidad."
    },
    {
        "id": 6, "domain": "Relational", "concept": "On (恩)",
        "E_base": "He felt a lifelong moral commitment to support his old mentor who helped him secure his first job.",
        "J_asym": "彼は、最初の仕事を得るのを助けてくれた古い恩師に対して、生涯にわたる恩を感じていた。",
        "E_lit": "He felt a lifelong structural debt of gratitude toward his old mentor who helped him secure his first job.",
        "S_sym": "Sintió un compromiso moral de por vida para apoyar a su antiguo mentor que lo ayudó a asegurar su primer trabajo."
    },

    # --- DOMAIN 2: SOCIAL SPACE & GROUP HARMONY MECHANICS ---
    {
        "id": 7, "domain": "Social Space", "concept": "Wa (和)",
        "E_base": "The team members chose to agree with the manager's proposal to prevent arguing during the open meeting.",
        "J_asym": "チームのメンバーは、公開ミーティングでの議論を避けるために、マネージャーの提案に同調して和を保つことを選んだ。",
        "E_lit": "The team members chose to conform to the manager's proposal to maintain harmony and avoid argument in the public meeting.",
        "S_sym": "Los miembros del equipo decidieron estar de acuerdo con la propuesta del gerente para evitar discusiones durante la reunión abierta."
    },
    {
        "id": 8, "domain": "Social Space", "concept": "Kuuki wo Yomu (空気を読む)",
        "E_base": "He remained completely silent during the tense boardroom disagreement, sensing the unspoken mood of the room.",
        "J_asym": "彼は、取締役会の緊張した不一致の最中に、周囲の空気を読んで、完全に沈黙を守ることにした。",
        "E_lit": "He chose to remain completely silent during the tense boardroom disagreement, carefully reading the unwritten atmospheric social signals.",
        "S_sym": "Permaneció en completo silencio durante la tensa discrepancia en la sala de juntas, sintiendo el estado de ánimo tácito de la sala."
    },
    {
        "id": 9, "domain": "Social Space", "concept": "Uchi-Soto (内-外)",
        "E_base": "The corporate team shared raw financial details internally but kept the information confidential from outside clients.",
        "J_asym": "企業チームは、内の身内には生の財務詳細を共有したが、外のクライアントには情報を秘密に保った。",
        "E_lit": "The corporate team shared raw financial details with the internal group but kept it strictly confidential from the outside domain.",
        "S_sym": "El equipo corporativo compartió detalles financieros crudos internamente pero mantuvo la información confidencial para los clientes externos."
    },
    {
        "id": 10, "domain": "Social Space", "concept": "Honne to Tatemae (本音と建前)",
        "E_base": "He praised the low-quality design to avoid hurting feelings, hiding his true personal complaints completely.",
        "J_asym": "彼は相手の感情を傷つけるのを避けるために、建前を使って低品質のデザインを褒め、本音を完全に隠した。",
        "E_lit": "He praised the low-quality design using public facade consensus while hiding his true intrinsic opinion completely.",
        "S_sym": "Elogió el diseño de baja calidad para evitar herir sentimientos, ocultando sus verdaderas quejas personales por completo."
    },
    {
        "id": 11, "domain": "Social Space", "concept": "Meiwaku (迷惑)",
        "E_base": "She stepped outside into the cold rain to take her phone call, preventing any disruption to the quiet café.",
        "J_asym": "彼女は静かなカフェの周囲に迷惑をかけるのを防ぐために、冷たい雨の屋外に出て電話に出た。",
        "E_lit": "She stepped outside into the cold rain to take her phone call to prevent causing collective social nuisance to the quiet café.",
        "S_sym": "Salió bajo la lluvia fría para atender su llamada telefónica, evitando cualquier interrupción en el café tranquilo."
    },
    {
        "id": 12, "domain": "Social Space", "concept": "Enryo (遠慮)",
        "E_base": "He declined the last serving of food on the shared platter, ensuring others had enough to eat.",
        "J_asym": "彼は、他の人々が十分に食べる時間を確保するために、遠慮の精神から大皿の最後の料理を辞退した。",
        "E_lit": "He declined the last serving of food on the shared platter out of structural self-restraint for the collective group.",
        "S_sym": "Rechazó la última porción de comida en el plato compartido, asegurando que otros tuvieran suficiente para comer."
    },

    # --- DOMAIN 3: EXISTENTIAL ORIENTATION & PERSEVERANCE DYNAMICS ---
    {
        "id": 13, "domain": "Existential", "concept": "Ganbaru (頑張る)",
        "E_base": "The engineer worked through the night on the broken software code, refusing to stop until it functioned perfectly.",
        "J_asym": "エンジニアは、完全に機能するまで諦めずに頑張る姿勢で、壊れたソフトウェアコードの修正に一晩中取り組んだ。",
        "E_lit": "The engineer worked through the night on the broken software code, stubbornly persisting and enduring hardships until it functioned perfectly.",
        "S_sym": "El ingeniero trabajó toda la noche en el código de software roto, negándose a detenerse hasta que funcionara perfectamente."
    },
    {
        "id": 14, "domain": "Existential", "concept": "Ikigai (生きがい)",
        "E_base": "The master craftsman woke up at dawn every day to shape clay, feeling a deep purpose in his life's work.",
        "J_asym": "その職人は、自分の生涯の仕事に深い生きがいを感じながら、毎日夜明けに起きて粘土を形作った。",
        "E_lit": "The master craftsman woke up at dawn every day to shape clay, feeling an intrinsic motivation that makes his life worth living.",
        "S_sym": "El maestro artesano se despertaba al amanecer todos los días para moldear arcilla, sintiendo un profundo propósito en el trabajo de su vida."
    },
    {
        "id": 15, "domain": "Existential", "concept": "Mono no Aware (物の哀れ)",
        "E_base": "Watching the autumn leaves fall slowly in the wind made him think about the impermanent nature of beautiful things.",
        "J_asym": "秋の葉が風にゆっくりと落ちるのを見て、彼は美しいものの儚い性質に対する物の哀れを感じた。",
        "E_lit": "Watching the autumn leaves fall slowly in the wind triggered a deep, empathetic awareness of the transience of all beautiful things.",
        "S_sym": "Ver las hojas de otoño caer lentamente con el viento lo hizo pensar en la naturaleza efímera de las cosas hermosas."
    },
    {
        "id": 16, "domain": "Existential", "concept": "Wabi-Sabi (侘寂)",
        "E_base": "He chose an unpolished, cracked ceramic bowl for the table, appreciating its weathered texture and natural flaws.",
        "J_asym": "彼は、経年変化した質感と自然な欠陥を評価する侘寂の視点から、未研磨でひび割れたセラミックのボウルをテーブルに選んだ。",
        "E_lit": "He chose an unpolished, cracked ceramic bowl for the table, finding aesthetic beauty in its weathered transience and structural imperfection.",
        "S_sym": "Eligió un tazón de cerámica agrietado y sin pulir para la mesa, apreciando su textura desgastada y sus defectos naturales."
    },
    {
        "id": 17, "domain": "Existential", "concept": "Shoganai (しょうがない)",
        "E_base": "When the unexpected storm canceled the long-awaited outdoor festival, they accepted the disruption without anger.",
        "J_asym": "予期せぬ嵐で待ちに待った野外フェスティバルが中止になったとき、彼らはしょうがないと静かに受け入れた。",
        "E_lit": "When the unexpected storm canceled the long-awaited outdoor festival, they accepted the disruption with stoic, fatalistic resignation as something that cannot be helped.",
        "S_sym": "Cuando la tormenta inesperada canceló el tan esperado festival al aire libre, aceptaron la interrupción sin enojo."
    },
    {
        "id": 18, "domain": "Existential", "concept": "Gaman (我慢)",
        "E_base": "The athlete continued running despite the intense muscle fatigue, suppressing the physical pain without complaining.",
        "J_asym": "そのアスリートは、文句を言わずに肉体的な痛みを隠して耐える我慢の精神で、激しい筋肉疲労にもかかわらず走り続けた。",
        "E_lit": "The athlete continued running despite the intense muscle fatigue, enduring the physical pain with quiet dignity and self-control without complaining.",
        "S_sym": "El atleta continuó corriendo a pesar de la intensa fatiga muscular, suprimiendo el dolor físico sin quejarse."
    },
    {
        "id": 19, "domain": "Existential", "concept": "Kodawari (こだわり)",
        "E_base": "The baker spent five extra minutes perfecting the shape of every loaf, demanding complete personal perfection.",
        "J_asym": "そのパン職人は、独自の強いこだわりから、完全な個人的な完璧さを求めて、すべてのパンの形状を仕上げるのに5分長く費やした。",
        "E_lit": "The baker spent five extra minutes shaping every loaf, driven by an uncompromising, obsessive pursuit of personal perfection in his craft.",
        "S_sym": "El panadero pasó cinco minutos adicionales perfeccionando la forma de cada hogaza, exigiendo una perfección personal completa."
    },
    {
        "id": 20, "domain": "Existential", "concept": "Omotenashi (おもてなし)",
        "E_base": "The innkeeper anticipated the guest's preference for hot tea, preparing it carefully before they even requested it.",
        "J_asym": "旅館の主人は、おもてなしの心から、ゲストが温かいお茶を好むことを予期し、相手が要求する前に注意深く準備した。",
        "E_lit": "The innkeeper anticipated the guest's preference for hot tea, preparing it meticulously with selfless, proactive hospitality before any explicit request.",
        "S_sym": "El posadero anticipó la preferencia del huésped por el té caliente, preparándolo cuidadosamente antes de que lo solicitaran."
    }
]
print(f"Data matrix expansion successful. Scaled to {len(refined_comprehensive_dataset)} concepts.")

Data matrix expansion successful. Scaled to 20 concepts.


In [ ]:
expanded_records = []

for m_info in models_to_test:

    print(f"\nRunning high-density manifold extraction for: {m_info['name']}")

    m = m_info["model"]

    tok = m_info["tokenizer"]

    for triplet in refined_comprehensive_dataset:

        print(f"Processing Concept: {triplet['concept']}")

        # Extract pooled hidden states
        v_base = extract_pooled_hidden_states(
            triplet["E_base"],
            m,
            tok
        )

        v_asym = extract_pooled_hidden_states(
            triplet["J_asym"],
            m,
            tok
        )

        v_lit = extract_pooled_hidden_states(
            triplet["E_lit"],
            m,
            tok
        )

        v_sym = extract_pooled_hidden_states(
            triplet["S_sym"],
            m,
            tok
        )

        v_scram = extract_pooled_hidden_states(
            triplet["J_asym"],
            m,
            tok,
            scramble=True
        )

        num_layers = v_base.shape[0]

        for l in range(num_layers):

            expanded_records.append({

                "model": m_info["name"],

                "triplet_id": triplet["id"],

                "domain": triplet["domain"],

                "concept": triplet["concept"],

                "layer": l,

                "dist_asym": cosine(
                    v_asym[l],
                    v_base[l]
                ),

                "dist_lit": cosine(
                    v_lit[l],
                    v_base[l]
                ),

                "dist_sym": cosine(
                    v_sym[l],
                    v_base[l]
                ),

                "dist_scram": cosine(
                    v_scram[l],
                    v_base[l]
                )
            })

# Convert to dataframe
df_sprint_master = pd.DataFrame(
    expanded_records
)

# Save locally
df_sprint_master.to_csv(
    "sprint_master_controlled_manifold_data.csv",
    index=False
)

print("\nExtraction complete.")

print(
    f"Saved {len(df_sprint_master)} layerwise vectors to disk."
)


Running high-density manifold extraction for: TinyLlama-1.1B
Processing Concept: Giri (義理)
Processing Concept: Ninjo (人情)
Processing Concept: En (縁)
Processing Concept: Amaeru (甘える)
Processing Concept: Omoiyari (思いやり)
Processing Concept: On (恩)
Processing Concept: Wa (和)
Processing Concept: Kuuki wo Yomu (空気を読む)
Processing Concept: Uchi-Soto (内-外)
Processing Concept: Honne to Tatemae (本音と建前)
Processing Concept: Meiwaku (迷惑)
Processing Concept: Enryo (遠慮)
Processing Concept: Ganbaru (頑張る)
Processing Concept: Ikigai (生きがい)
Processing Concept: Mono no Aware (物の哀れ)
Processing Concept: Wabi-Sabi (侘寂)
Processing Concept: Shoganai (しょうがない)
Processing Concept: Gaman (我慢)
Processing Concept: Kodawari (こだわり)
Processing Concept: Omotenashi (おもてなし)

Running high-density manifold extraction for: Qwen-1.8B
Processing Concept: Giri (義理)
Processing Concept: Ninjo (人情)
Processing Concept: En (縁)
Processing Concept: Amaeru (甘える)
Processing Concept: Omoiyari (思いやり)
Processing Concept: On (恩)
Processing 

In [ ]:
print(df_sprint_master.head())

            model  triplet_id      domain    concept  layer  dist_asym  \
0  TinyLlama-1.1B           1  Relational  Giri (義理)      0   0.860650   
1  TinyLlama-1.1B           1  Relational  Giri (義理)      1   0.227023   
2  TinyLlama-1.1B           1  Relational  Giri (義理)      2   0.189004   
3  TinyLlama-1.1B           1  Relational  Giri (義理)      3   0.008156   
4  TinyLlama-1.1B           1  Relational  Giri (義理)      4   0.015015   

   dist_lit  dist_sym  dist_scram  
0  0.344130  0.648161    0.860650  
1  0.042497  0.094117    0.240743  
2  0.041951  0.108263    0.200424  
3  0.000719  0.001631    0.008352  
4  0.001537  0.003052    0.014440  


In [ ]:
print(df_sprint_master.shape)

(960, 9)


In [ ]:
print(
    df_sprint_master["model"].value_counts()
)

model
Qwen-1.8B         500
TinyLlama-1.1B    460
Name: count, dtype: int64


In [ ]:
print(
    df_sprint_master["concept"].nunique()
)

20


In [ ]:
print(
    df_sprint_master["domain"].value_counts()
)

domain
Existential     384
Relational      288
Social Space    288
Name: count, dtype: int64


In [ ]:
def run_permutation_test(
    group_asym,
    group_control,
    num_permutations=10000
):

    observed_diff = np.abs(

        np.mean(group_asym)
        -
        np.mean(group_control)

    )

    combined = np.concatenate([
        group_asym,
        group_control
    ])

    n_asym = len(group_asym)

    extreme_count = 0

    for _ in range(num_permutations):

        shuffled = np.random.permutation(
            combined
        )

        fake_asym = shuffled[:n_asym]

        fake_control = shuffled[n_asym:]

        fake_diff = np.abs(

            np.mean(fake_asym)
            -
            np.mean(fake_control)

        )

        if fake_diff >= observed_diff:

            extreme_count += 1

    p_value = extreme_count / num_permutations

    return p_value

print("Permutation testing module compiled.")

Permutation testing module compiled.


In [ ]:
tiny_zone = df_sprint_master[

    (df_sprint_master["model"] == "TinyLlama-1.1B")

    &

    (df_sprint_master["layer"].between(11, 18))

]

qwen_zone = df_sprint_master[

    (df_sprint_master["model"] == "Qwen-1.8B")

    &

    (df_sprint_master["layer"].between(12, 20))

]

print("Semantic zones isolated.")

Semantic zones isolated.


In [ ]:
tiny_p_lit = run_permutation_test(

    tiny_zone["dist_asym"].values,

    tiny_zone["dist_lit"].values

)

tiny_p_sym = run_permutation_test(

    tiny_zone["dist_asym"].values,

    tiny_zone["dist_sym"].values

)

print("=== TINYLLAMA PERMUTATION TESTS ===\n")

print(
    f"Asym vs Literal : p = {tiny_p_lit}"
)

print(
    f"Asym vs Spanish : p = {tiny_p_sym}"
)

=== TINYLLAMA PERMUTATION TESTS ===

Asym vs Literal : p = 0.0
Asym vs Spanish : p = 0.0


In [ ]:
qwen_p_lit = run_permutation_test(

    qwen_zone["dist_asym"].values,

    qwen_zone["dist_lit"].values

)

qwen_p_sym = run_permutation_test(

    qwen_zone["dist_asym"].values,

    qwen_zone["dist_sym"].values

)

print("=== QWEN PERMUTATION TESTS ===\n")

print(
    f"Asym vs Literal : p = {qwen_p_lit}"
)

print(
    f"Asym vs Spanish : p = {qwen_p_sym}"
)

=== QWEN PERMUTATION TESTS ===

Asym vs Literal : p = 0.0
Asym vs Spanish : p = 0.0


In [ ]:
models_to_test = [

    {
        "name": "TinyLlama-1.1B",
        "model": tiny_model,
        "tokenizer": tiny_tokenizer
    },

    {
        "name": "Qwen-1.8B",
        "model": qwen_model,
        "tokenizer": qwen_tokenizer
    }
]

print("Model registry initialized successfully.")

Model registry initialized successfully.


In [ ]:
df_master = pd.read_csv(
    "sprint_master_controlled_manifold_data.csv"
)

print(df_master.shape)

(960, 9)


In [ ]:
import statsmodels.api as sm

from statsmodels.formula.api import ols

In [ ]:
tl_zone = df_master[

    (df_master["model"] == "TinyLlama-1.1B")

    &

    (df_master["layer"].between(11, 18))

]

qw_zone = df_master[

    (df_master["model"] == "Qwen-1.8B")

    &

    (df_master["layer"].between(12, 20))

]

combined_zone_df = pd.concat([
    tl_zone,
    qw_zone
])

print(combined_zone_df.shape)

(340, 9)


In [ ]:
melted_zone = pd.melt(

    combined_zone_df,

    id_vars=[
        "model",
        "concept",
        "layer"
    ],

    value_vars=[
        "dist_asym",
        "dist_lit",
        "dist_sym"
    ],

    var_name="prompt_type",

    value_name="cosine_distance"
)

print(melted_zone.head())

            model    concept  layer prompt_type  cosine_distance
0  TinyLlama-1.1B  Giri (義理)     11   dist_asym         0.134622
1  TinyLlama-1.1B  Giri (義理)     12   dist_asym         0.151953
2  TinyLlama-1.1B  Giri (義理)     13   dist_asym         0.172544
3  TinyLlama-1.1B  Giri (義理)     14   dist_asym         0.201895
4  TinyLlama-1.1B  Giri (義理)     15   dist_asym         0.224945


In [ ]:
anova_model = ols(

    'cosine_distance ~ C(model) + C(prompt_type)',

    data=melted_zone

).fit()

print("ANOVA model fitted successfully.")

ANOVA model fitted successfully.


In [ ]:
anova_table = sm.stats.anova_lm(
    anova_model,
    typ=2
)

print(anova_table)

                  sum_sq      df           F         PR(>F)
C(model)        0.476831     1.0   91.605183   7.752532e-21
C(prompt_type)  6.681377     2.0  641.787484  6.095886e-181
Residual        5.288572  1016.0         NaN            NaN


In [ ]:
total_ss = anova_table["sum_sq"].sum()

anova_table["eta_squared"] = (

    anova_table["sum_sq"]

    /

    total_ss
)

print(
    anova_table[
        ["df", "sum_sq", "F", "PR(>F)", "eta_squared"]
    ]
)

                    df    sum_sq           F         PR(>F)  eta_squared
C(model)           1.0  0.476831   91.605183   7.752532e-21     0.038310
C(prompt_type)     2.0  6.681377  641.787484  6.095886e-181     0.536796
Residual        1016.0  5.288572         NaN            NaN     0.424895
